# RNN用系列を非線形ARへ変換

`ISLP23-C10-Q11` · ISLP 2023 · Chapter 10, Exercise 11

## この問題で行うこと

前問Q10と同じNYSEの短い時系列窓を平坦化し、今度は非線形モデルで自己回帰する。Q10の線形ARと同じ分割でテストR²を比較する。

[原典の設問・正確な条件を確認する](https://drive.google.com/file/d/1ajFkHO6zjrdGNqhqW1jKBZdiNGh_8YQ1/view)

この説明は学習用の要約です。数値条件・指定手法・各小問は原典に従ってください。

## 進め方

1. 原典を開いて、このIDの設問と使用データを確認する。
2. データの行数・列・目的変数を確認してから、下に実装する。
3. 指定された指標・図・考察を残し、再実行して結果が再現するか確かめる。

Colabにデータがない場合は原典の配布元から取得してください。

In [ ]:
# まず必要なライブラリ・データを読み込み、形状と欠損を確認する
import numpy as np
import pandas as pd

# 例: data = ...  # 原典の指定データ
# display(data.head())
# print(data.shape)


## 前問Q10とのつながり

Q10で作ったNYSEの短い時系列窓・目的変数・訓練/テスト分割をそのまま使います。窓を2次元の特徴行列へ平坦化した後、線形回帰の代わりに非線形回帰器を学習します。**同じテスト集合のR²**でQ10の線形ARと比較してください。時系列の未来情報が訓練側へ混ざらないよう、分割順序も維持します。

In [ ]:
# ColabでISLPが未導入なら、次の行を一度実行する
# %pip -q install ISLP
from ISLP import load_data
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

nyse = load_data('NYSE')
series_cols = ['DJ_return', 'log_volume', 'log_volatility']
windows = np.stack(
    [nyse[series_cols].shift(lag).to_numpy() for lag in range(5, 0, -1)],
    axis=1,
)
valid = np.isfinite(windows).all(axis=(1, 2))
X_seq = windows[valid]
y = nyse.loc[valid, 'log_volume'].to_numpy()
is_train = nyse.loc[valid, 'train'].astype(bool).to_numpy()
X_flat = X_seq.reshape(len(X_seq), -1)
scaler = StandardScaler().fit(X_flat[is_train])
X_train = scaler.transform(X_flat[is_train])
X_test = scaler.transform(X_flat[~is_train])
y_train, y_test = y[is_train], y[~is_train]
print('train/test:', X_train.shape, X_test.shape)


## 実装チェック

- 同じ `X_train` / `X_test` で線形モデルと非線形モデルを学習する。
- 非線形モデルの構造と調整方法を記録する。調整にテスト集合は使わない。
- 両者のテストR²を同じ表に出し、差の理由を考察する。

## 解答

[原典の解答に対応する指示](https://drive.google.com/file/d/1ajFkHO6zjrdGNqhqW1jKBZdiNGh_8YQ1/view)を確認し、求められる計算・図表・説明をここで完成させる。

In [ ]:
# 解答 の実装


**結果・判断の根拠：**

